In [1]:
# Outlier exploration; see docs/project-update/features/03-outlier-zh.md
# Starting point: flag_anomalies(raw) -> fill_missing. This notebook does not run 01 and 02 itself:
# run them first in the same kernel (04 does: %run 01, %run 02, then %run -i 03; -i lets this notebook see their functions).
df = fill_missing(flag_anomalies(data), sector_medians(data))
y = df["def_date"].notna()
df.shape

(1023552, 46)

In [ ]:
# Outlier handling; see docs/project-update/features/03-outlier-zh.md
# Placeholder: no handling for now because we expect a tree model. For a linear model, winsorize here at the training set's p1/p99.
def handle_outliers(df):
    """Take filled data, return it with outliers handled. Currently returns the data unchanged."""
    return df

In [2]:
# Amount columns: tails, skewness and share of negatives
AMOUNT = [*B_COLS, "asst_tot", "wc_net", "margin_fin"]
q = df[AMOUNT].quantile([0, 0.01, 0.5, 0.99, 1]).T.set_axis(["min", "p1", "p50", "p99", "max"], axis=1)
q["max/p99"] = q["max"] / q["p99"]
q["skew"] = df[AMOUNT].skew()
q["pct_neg"] = (df[AMOUNT] < 0).mean() * 100
q.round(2).sort_values("skew", ascending=False)

,min,p1,p50,p99,max,max/p99,skew,pct_neg
exp_financing,-5.307809e+06,0.00,38671.00,1.551257e+06,4.389405e+09,2829.58,581.28,0.07
liab_lt,-1.283245e+06,0.00,0.00,3.057861e+06,2.707827e+10,8855.30,557.80,0.09
AP_lt,-5.879780e+05,0.00,0.00,2.000000e+05,7.332880e+08,3666.44,419.02,0.01
cash_and_equiv,-2.584593e+07,0.00,49644.50,5.836934e+06,5.519622e+09,945.64,358.41,0.21
asst_tang_fixed,-1.711468e+06,0.00,595856.50,3.232902e+07,5.822144e+10,1800.90,340.57,0.03
eqty_tot,-8.052920e+08,-426628.12,718923.50,3.700596e+07,3.356488e+10,907.01,305.01,3.98
asst_tot,1.500000e+06,1520050.02,3458428.50,1.034126e+08,7.164463e+10,692.80,257.31,0.00
debt_fin_st,-1.458602e+07,0.00,0.00,1.281425e+06,3.713192e+09,2897.70,219.87,0.03
debt_fin_lt,-5.449856e+06,0.00,0.00,1.445236e+06,2.229168e+09,1542.42,218.49,0.01
liab_lt_emp,-3.373560e+05,0.00,34900.00,2.190477e+06,1.734028e+09,791.62,211.05,0.02


In [3]:
# Most of the spread is firm size: the same columns divided by asst_tot (always > 0)
ratio = df[[c for c in AMOUNT if c != "asst_tot"]].div(df["asst_tot"], axis=0)
rq = ratio.quantile([0, 0.01, 0.5, 0.99, 1]).T.set_axis(["min", "p1", "p50", "p99", "max"], axis=1)
rq["skew"] = ratio.skew()
rq.round(3).sort_values("skew", ascending=False)

,min,p1,p50,p99,max,skew
cf_operations,-73.984,-0.217,0.022,0.277,489.253,689.186
exp_financing,-0.642,0.000,0.011,0.060,31.911,388.208
COGS,-1.477,0.000,0.547,3.682,1307.549,382.272
rev_operating,-3.072,0.000,0.574,3.728,820.561,314.635
inc_financing,-4.166,-0.058,-0.009,0.051,21.130,163.295
debt_fin_st,-4.184,0.000,0.000,0.197,23.357,139.185
debt_lt,-0.621,0.000,0.095,0.971,96.927,62.309
AP_lt,-0.138,0.000,0.000,0.031,3.250,26.110
liab_lt,-0.598,0.000,0.000,0.195,9.035,21.127
debt_fin_lt,-0.694,0.000,0.000,0.282,6.497,12.954


In [4]:
# Do the tails carry default signal? Default rate in the bottom 1%, middle 98% and top 1% of each ratio
def tail_rates(x):
    lo, hi = x.quantile([0.01, 0.99])
    return pd.Series({"bottom_1%": y[x < lo].mean(), "middle": y[x.between(lo, hi)].mean(), "top_1%": y[x > hi].mean()})

ratio.apply(tail_rates).T.mul(100).round(2).sort_values("bottom_1%", ascending=False)

,bottom_1%,middle,top_1%
eqty_tot,40.80,4.02,2.59
profit,36.80,4.04,4.32
cf_operations,36.42,4.04,4.14
taxes,35.87,4.05,4.18
prof_operations,34.37,4.07,3.33
ebitda,34.29,4.08,3.30
goodwill,28.65,4.12,4.44
inc_financing,21.98,4.21,2.77
inc_extraord,16.35,4.18,10.88
margin_fin,15.48,4.25,5.02


In [5]:
# roa: extreme values (formula matches exactly, see 01-financial-anomalies.md)
bins = pd.cut(df["roa"], [-float("inf"), -100, -50, -20, 0, 20, 50, 100, float("inf")])
df.groupby(bins, observed=True).agg(n=("roa", "size"), default_rate=("def_date", lambda s: s.notna().mean() * 100)).round(2)

,n,default_rate
roa,,
"(-inf, -100.0]",784,50.38
"(-100.0, -50.0]",2314,43.86
"(-50.0, -20.0]",9625,28.30
"(-20.0, 0.0]",231959,6.44
"(0.0, 20.0]",749995,3.33
"(20.0, 50.0]",26577,1.85
"(50.0, 100.0]",2198,6.55
"(100.0, inf]",100,17.00


In [6]:
# rev_operating == 0: revenue-denominator ratios break; who are these firms?
zero = df["rev_operating"] == 0
print(f"{zero.sum()} rows, default rate {y[zero].mean():.2%} vs {y[~zero].mean():.2%}")
df[zero]["ateco_sector"].value_counts().head(10)

31370 rows, default rate 5.73% vs 4.33%


ateco_sector
68.0    9730
41.0    6519
70.0    6096
35.0    1173
55.0     895
46.0     672
1.0      567
82.0     357
71.0     286
47.0     265
Name: count, dtype: int64